In [ ]:
!pip install trl peft bitsandbytes transformers gguf -U 

# 准备数据

## 加载数据

In [4]:

from datasets import load_dataset

# 加载数据集
dataset = load_dataset("Magpie-Align/Magpie-Qwen2-Pro-1M-v0.1")
dataset = dataset["train"]


README.md:   0%|          | 0.00/6.19k [00:00<?, ?B/s]

train-00000-of-00009.parquet:   0%|          | 0.00/232M [00:00<?, ?B/s]

train-00001-of-00009.parquet:   0%|          | 0.00/231M [00:00<?, ?B/s]

train-00002-of-00009.parquet:   0%|          | 0.00/232M [00:00<?, ?B/s]

train-00003-of-00009.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

...

train-00007-of-00009.parquet:   0%|          | 0.00/249M [00:00<?, ?B/s]

train-00008-of-00009.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 900000
    })
    test: Dataset({
        features: ['text'],
        num_rows: 100000
    })
})


## 数据清洗

In [ ]:
# 格式化数据集
def format_instruction(example):
    return {
        "text": (
            "<|user|>\n"
            f"{example['instruction']}\n"
            "<|end|>\n"
            "<|assistant|>\n"
            f"{example['response']}\n"
            "<|end|>"
        )
    }

# 直接使用 dataset.column_names
formatted_dataset = dataset.map(format_instruction, batched=False, remove_columns=dataset.column_names)

# 训练-测试集划分
formatted_dataset = formatted_dataset.train_test_split(test_size=0.1)  # 90% 训练集, 10% 测试集

# 输出数据集信息
print(formatted_dataset)

# 模型准备

## 加载量化模型

Quantize the model in FP4

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


model_id="/kaggle/input/qwen2.5/gguf/3b-instruct/1"

tokenizer=AutoTokenizer.from_pretrained(model_id, gguf_file="qwen2.5-3b-instruct-q4_k_m.gguf", trust_remote_code=True)

base_model=AutoModelForCausalLM.from_pretrained(
    model_id,
    gguf_file="qwen2.5-3b-instruct-q4_k_m.gguf",
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

base_model.config.use_cache=False
print(base_model)

Converting and de-quantizing GGUF tensors...:   0%|          | 0/435 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151643)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=9.999999974752427e-07)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=9.999999974752427e-07)
      )
    )
    (norm)

## 调整分词规则

In [ ]:
tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="right" # this fixed the weird overflow issue with fp16 training
# 添加自定义标记
CUSTOM_TOKENS = ["<think>", "</think>"]
tokenizer.add_special_tokens({"additional_special_tokens": CUSTOM_TOKENS})

In [8]:
base_model.resize_token_embeddings(len(tokenizer))  # 调整以适应自定义标记

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(151938, 2048, padding_idx=151643)

# 模型训练

## 低秩优化

In [9]:
from peft import LoraConfig, TaskType
from peft import AutoPeftModelForCausalLM

peft_config=LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj","v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

peft_config

LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'v_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, eva_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)

## 训练参数

In [ ]:
from trl import SFTConfig, SFTTrainer


training_args=SFTConfig(
    output_dir="./distill",
    max_steps=10,
    logging_steps=1,
    save_steps=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    group_by_length=False,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    weight_decay=0.05,
    optim="paged_adamw_32bit",
    fp16=True,
    remove_unused_columns=False,
    report_to="none" #wandb/none
)

sft_trainer=SFTTrainer(
    model=base_model,
    train_dataset=formatted_dataset['train'],
    eval_dataset=formatted_dataset['test'],
    peft_config=peft_config,
    #packing=True,
    #max_seq_length=None,
    tokenizer=tokenizer,
    args=training_args,
)

/tmp/ipykernel_156/299063248.py:25: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  sft_trainer=SFTTrainer(


Converting train dataset to ChatML:   0%|          | 0/900000 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/900000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/900000 [00:00<?, ? examples/s]

In [ ]:
sft_trainer.train()

## 合并并保存模型

In [ ]:
import gc

del sft_trainer, base_model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from peft import AutoPeftModelForCausalLM

model=AutoPeftModelForCausalLM.from_pretrained("./sft/checkpoint-10", device_map="auto", torch_dtype=torch.bfloat16)
model=model.merge_and_unload()

model.save_pretrained("./sft/final_merged_checkpoint", safe_serialization=True)

## 模型推理

In [ ]:
from transformers import pipeline

# 加载微调后的模型
model = AutoModelForCausalLM.from_pretrained(
    "./phi-3-deepseek-finetuned-final",
    device_map="auto",
    torch_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained("./phi-3-deepseek-finetuned-final")
model.resize_token_embeddings(len(tokenizer))

# 创建聊天流程
chat_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

# 生成响应
prompt = """<|user|>
What's the probability of rolling a 7 with two dice?
<|end|>
<|assistant|>
"""

output = chat_pipeline(
    prompt,
    max_new_tokens=5000,
    temperature=0.7,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id
)

print(output[0]['generated_text'])
